[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/distributed_training.ipynb)

# 02 . Distributed Training

## Concept — Distributed Training (DataParallel, DDP, FSDP) (revision notes)

**What it is**
- Splitting training across several processes/devices so you can process more data per step (data parallelism) or fit a model that does not fit on one device (sharding).
- `nn.DataParallel`: legacy single-process, multi-thread; `DistributedDataParallel` (DDP): one process per device, gradients averaged with all-reduce; `FullyShardedDataParallel` (FSDP): shards parameters, gradients and optimizer state across ranks.

**Why it matters**
- Training throughput scales almost linearly with DDP when communication overlaps with backward.
- FSDP lets models far larger than one GPU's memory train by keeping only a 1/N shard of the state per rank.
- Subtle bugs (unsynced seeds, duplicated data, rank-0-only logic) silently ruin results, so understanding the mechanics matters.

**When to use**
- DDP: the model fits on one GPU and you have 2+ GPUs (default choice, even on one machine).
- FSDP: model + optimizer state exceed one GPU's memory.
- DataParallel: basically never; it is slower than DDP (GIL, replicating the model every step) and single-node only.

**Math & intuition**
- Each of N ranks computes grads on its own mini-batch shard; all-reduce(mean) gives every rank the same gradient equal to the gradient of the average loss over the global batch. Identical init + identical updates keep replicas identical.
- Memory per rank with Adam in fp32: DDP ~ 16 bytes/param (4 weights + 4 grads + 8 optimizer); FSDP/ZeRO-3 ~ 16/N bytes/param plus transient all-gathered layers.
- Backends: `nccl` for GPU, `gloo` for CPU (what we use here).

### 1. Environment variables and ranks: the vocabulary

**What this cell does (plain language):** we list the environment variables that launchers (`torchrun`) set for each process — `RANK`, `WORLD_SIZE`, `LOCAL_RANK`, `MASTER_ADDR`, `MASTER_PORT` — and check what this notebook's single process sees, plus which backends are compiled in.

In [1]:
import os, sys, tempfile, subprocess, textwrap
import torch, torch.nn as nn, torch.distributed as dist

for k in ("RANK", "WORLD_SIZE", "LOCAL_RANK", "MASTER_ADDR", "MASTER_PORT"):
    print(f"{k:12s} = {os.environ.get(k)}")      # None here: we are NOT inside a launcher
print("dist available:", dist.is_available(), "| gloo:", dist.is_gloo_available(), "| nccl:", dist.is_nccl_available())
print("dist initialized in this process:", dist.is_initialized())

RANK         = None
WORLD_SIZE   = None
LOCAL_RANK   = None
MASTER_ADDR  = None
MASTER_PORT  = None
dist available: True | gloo: True | nccl: True
dist initialized in this process: False


### 2. DataParallel: what it does (and why not to use it)

**What this cell does (plain language):** we wrap a model in `nn.DataParallel`. With no GPUs it simply runs the module as-is, so we can only illustrate the API; the key point in the printed text is the design: one process, scatter inputs along dim 0, replicate the model per forward, gather outputs on device 0.

In [2]:
torch.manual_seed(0)
net = nn.Linear(4, 2)
dp = nn.DataParallel(net)                 # with 0 GPUs this just forwards to `net`
print("visible CUDA devices:", torch.cuda.device_count())
out = dp(torch.randn(8, 4))
print("output shape:", tuple(out.shape), "| params live under dp.module:", type(dp.module).__name__)
print(textwrap.dedent('''
    DataParallel each step: scatter batch -> replicate model to every GPU -> parallel forward
    -> gather outputs on GPU0 -> backward -> reduce grads to GPU0 -> update on GPU0.
    Problems: GIL contention, GPU0 memory/compute hot-spot, model re-copied each step, no multi-node.
'''))

visible CUDA devices: 0
output shape: (8, 2) | params live under dp.module: Linear

DataParallel each step: scatter batch -> replicate model to every GPU -> parallel forward
-> gather outputs on GPU0 -> backward -> reduce grads to GPU0 -> update on GPU0.
Problems: GIL contention, GPU0 memory/compute hot-spot, model re-copied each step, no multi-node.



### 3. A collective by hand: all_reduce in a 2-process gloo group

**What this cell does (plain language):** we write a small script to a temp directory and launch it with `torch.multiprocessing.spawn` in a subprocess (spawning from inside a notebook cell is fragile because workers must import the target function from a real file). Each rank starts with a different tensor, calls `all_reduce(SUM)`, and ends with the identical total — the primitive DDP is built on. The helper `run_script` we define here is reused for the DDP demo.

In [3]:
workdir = tempfile.mkdtemp(prefix="ddp_demo_")

def run_script(name, source, timeout=120):
    # Write source to workdir/name and run it in a fresh interpreter; return its stdout
    path = os.path.join(workdir, name)
    with open(path, "w") as f: f.write(textwrap.dedent(source))
    r = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0: print("STDERR:", r.stderr[-2000:])
    return r.stdout

allreduce_src = '''
import os, torch, torch.distributed as dist, torch.multiprocessing as mp

def worker(rank, world_size):
    os.environ["MASTER_ADDR"] = "127.0.0.1"; os.environ["MASTER_PORT"] = "29511"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    t = torch.tensor([float(rank + 1)] * 3)          # rank0: [1,1,1]  rank1: [2,2,2]
    before = t.tolist()
    dist.all_reduce(t, op=dist.ReduceOp.SUM)         # in-place sum across ranks
    for r in range(world_size):                      # take turns printing so lines don't interleave
        if r == rank: print(f"rank {rank}: before={before} after all_reduce={t.tolist()}", flush=True)
        dist.barrier()
    dist.destroy_process_group()

if __name__ == "__main__":
    mp.spawn(worker, args=(2,), nprocs=2, join=True)
'''
print(run_script("allreduce_demo.py", allreduce_src))

rank 0: before=[1.0, 1.0, 1.0] after all_reduce=[3.0, 3.0, 3.0]
rank 1: before=[2.0, 2.0, 2.0] after all_reduce=[3.0, 3.0, 3.0]



### 4. Real DDP on CPU: training two replicas on different data shards

**What this cell does (plain language):** we launch a 2-process gloo DDP job. Both ranks build the model with the SAME seed (so replicas start identical), wrap it in `DistributedDataParallel`, and each trains on a DIFFERENT half of a synthetic dataset. We save per-rank results to JSON files so the notebook can inspect them afterward. Gradient sync is verified in the next cell.

In [4]:
ddp_src = '''
import os, json, sys, torch, torch.nn as nn, torch.distributed as dist, torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

OUT = sys.argv[1]

def make_data():
    g = torch.Generator().manual_seed(123)
    X = torch.randn(64, 8, generator=g); W = torch.randn(8, 1, generator=g)
    return X, X @ W

def worker(rank, world_size):
    os.environ["MASTER_ADDR"] = "127.0.0.1"; os.environ["MASTER_PORT"] = "29512"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    torch.manual_seed(0)                              # SAME init on every rank
    model = nn.Sequential(nn.Linear(8, 16), nn.Tanh(), nn.Linear(16, 1))
    ddp = DDP(model)                                  # broadcasts rank0 params + hooks grad all-reduce
    opt = torch.optim.SGD(ddp.parameters(), lr=0.1)
    X, Y = make_data()
    xs, ys = X[rank::world_size], Y[rank::world_size] # rank r gets every world_size-th sample
    losses = []
    for step in range(30):
        opt.zero_grad()
        loss = nn.functional.mse_loss(ddp(xs), ys)    # local loss on local shard
        loss.backward()                               # <- grads are all-reduced (averaged) here
        if step == 0:                                 # save step-0 synced grads for verification
            g0 = torch.cat([p.grad.flatten() for p in model.parameters()])
            torch.save(g0, os.path.join(OUT, f"grad0_rank{rank}.pt"))
        opt.step(); losses.append(loss.item())
    flat = torch.cat([p.detach().flatten() for p in model.parameters()])
    torch.save(flat, os.path.join(OUT, f"params_rank{rank}.pt"))
    json.dump(losses, open(os.path.join(OUT, f"losses_rank{rank}.json"), "w"))
    dist.barrier(); dist.destroy_process_group()

if __name__ == "__main__":
    mp.spawn(worker, args=(2,), nprocs=2, join=True)
'''
outdir = os.path.join(workdir, "out"); os.makedirs(outdir, exist_ok=True)
path = os.path.join(workdir, "ddp_train.py")
with open(path, "w") as f: f.write(textwrap.dedent(ddp_src))
r = subprocess.run([sys.executable, path, outdir], capture_output=True, text=True, timeout=120)
print("exit code:", r.returncode); print(r.stderr[-1500:] if r.returncode else "")
print(sorted(os.listdir(outdir)))

exit code: 0

['grad0_rank0.pt', 'grad0_rank1.pt', 'losses_rank0.json', 'losses_rank1.json', 'params_rank0.pt', 'params_rank1.pt']


### 5. Verify gradient synchronization

**What this cell does (plain language):** we load the step-0 gradients and final parameters saved by each rank and check three things: (1) both ranks have IDENTICAL gradients even though they saw different data; (2) that gradient equals the average of the two single-process shard gradients that we recompute here without any distributed code; (3) final parameters are identical, so replicas never drifted apart.

In [5]:
import json
g0 = [torch.load(os.path.join(outdir, f"grad0_rank{r}.pt")) for r in (0, 1)]
p  = [torch.load(os.path.join(outdir, f"params_rank{r}.pt")) for r in (0, 1)]
print("step-0 grads identical across ranks :", torch.allclose(g0[0], g0[1], atol=1e-7))
print("final params identical across ranks :", torch.allclose(p[0], p[1], atol=1e-7))

# Recompute reference gradients in ONE process: average of per-shard gradients
g = torch.Generator().manual_seed(123)
X = torch.randn(64, 8, generator=g); W = torch.randn(8, 1, generator=g); Y = X @ W
def shard_grad(r):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(8, 16), nn.Tanh(), nn.Linear(16, 1))
    nn.functional.mse_loss(m(X[r::2]), Y[r::2]).backward()
    return torch.cat([q.grad.flatten() for q in m.parameters()])
ref = (shard_grad(0) + shard_grad(1)) / 2
print("DDP grad == mean of shard grads     :", torch.allclose(g0[0], ref, atol=1e-6))
# Different shards -> the *local* gradients really are different:
print("local shard grads differ            :", not torch.allclose(shard_grad(0), shard_grad(1)))

step-0 grads identical across ranks : True
final params identical across ranks : True
DDP grad == mean of shard grads     : True
local shard grads differ            : True


### 6. Loss curves from both ranks

**What this cell does (plain language):** we read the per-rank loss histories. Each rank reports its LOCAL loss (on its own shard), so the two curves are similar but not identical, even though the model weights are identical — a common source of confusion when logging from DDP.

In [6]:
L = [json.load(open(os.path.join(outdir, f"losses_rank{r}.json"))) for r in (0, 1)]
for step in (0, 10, 20, 29):
    print(f"step {step:2d}: rank0 loss={L[0][step]:.4f}  rank1 loss={L[1][step]:.4f}")
print("loss decreased on both ranks:", all(l[-1] < l[0] for l in L))

step  0: rank0 loss=6.1536  rank1 loss=3.9405
step 10: rank0 loss=0.1215  rank1 loss=0.0927
step 20: rank0 loss=0.0804  rank1 loss=0.0551
step 29: rank0 loss=0.0685  rank1 loss=0.0458
loss decreased on both ranks: True


### 7. DistributedSampler: giving each rank a different shard

**What this cell does (plain language):** `DistributedSampler` does the sharding for a real `DataLoader`. We pass `num_replicas` and `rank` explicitly, so we can show what ranks 0 and 1 would each iterate over without launching anything. `set_epoch` changes the shuffle each epoch; forgetting it means every epoch uses the same order.

In [7]:
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
ds = TensorDataset(torch.arange(10))
for rank in (0, 1):
    s = DistributedSampler(ds, num_replicas=2, rank=rank, shuffle=True, seed=0)
    for epoch in (0, 1):
        s.set_epoch(epoch)                       # must be called every epoch
        print(f"rank {rank} epoch {epoch}: indices {list(s)}")

rank 0 epoch 0: indices [4, 7, 3, 0, 6]
rank 0 epoch 1: indices [5, 1, 0, 9, 7]
rank 1 epoch 0: indices [1, 5, 9, 8, 2]
rank 1 epoch 1: indices [6, 2, 8, 3, 4]


### 8. FSDP explained: sharding arithmetic

**What this cell does (plain language):** FSDP needs real multi-process and (usually) GPUs, so here we just compute what it buys you. For a model with P parameters trained with Adam in fp32, we tabulate per-rank state memory for DDP (everything replicated) vs FSDP (everything sharded) as the number of ranks grows.

In [8]:
P = 1_000_000_000                       # a 1B-parameter model
bytes_per_param = 4 + 4 + 8             # weights + grads + Adam (exp_avg, exp_avg_sq)
print(f"{'ranks':>5} | {'DDP GB/rank':>12} | {'FSDP GB/rank (state)':>20}")
for n in (1, 2, 4, 8, 64):
    ddp_gb = P * bytes_per_param / 1e9
    fsdp_gb = P * bytes_per_param / n / 1e9
    print(f"{n:>5} | {ddp_gb:>12.1f} | {fsdp_gb:>20.2f}")
print("FSDP also needs memory for the all-gathered full weights of the layer currently executing.")

ranks |  DDP GB/rank | FSDP GB/rank (state)
    1 |         16.0 |                16.00
    2 |         16.0 |                 8.00
    4 |         16.0 |                 4.00
    8 |         16.0 |                 2.00
   64 |         16.0 |                 0.25
FSDP also needs memory for the all-gathered full weights of the layer currently executing.


### 9. Reference code (not executed): torchrun, DDP and FSDP templates

**What this cell does (plain language):** shows the production templates you would run on real GPUs. It is deliberately NOT executed here (no GPUs), but the code is complete enough to adapt. Launch with `torchrun --nproc_per_node=4 train.py`.

In [9]:
# ---- train.py (illustrative; run with: torchrun --nproc_per_node=4 train.py) ----
# import os, torch, torch.distributed as dist
# from torch.nn.parallel import DistributedDataParallel as DDP
# from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
# from torch.utils.data.distributed import DistributedSampler
#
# dist.init_process_group("nccl")                     # torchrun sets RANK/WORLD_SIZE/MASTER_*
# local_rank = int(os.environ["LOCAL_RANK"])
# torch.cuda.set_device(local_rank)
# model = MyModel().cuda()
# model = DDP(model, device_ids=[local_rank])          # or: FSDP(model)  for sharding
# sampler = DistributedSampler(dataset, shuffle=True)
# loader = DataLoader(dataset, batch_size=32, sampler=sampler)
# for epoch in range(epochs):
#     sampler.set_epoch(epoch)
#     for x, y in loader:
#         ...                                          # usual loop
# if dist.get_rank() == 0:                             # save from ONE rank only
#     torch.save(model.module.state_dict(), "ckpt.pt") # model.module for DDP; FSDP needs state_dict_type context
# dist.destroy_process_group()
print("template only - see comments above")

template only - see comments above


### 10. Cleanup

**What this cell does (plain language):** we delete the temp directory the demos wrote to, and confirm no process group lingers in this notebook's process (the group lived only in the spawned workers).

In [10]:
import shutil
shutil.rmtree(workdir, ignore_errors=True)
print("temp dir removed:", not os.path.exists(workdir), "| group initialized here:", dist.is_initialized())

temp dir removed: True | group initialized here: False


## Common bugs

- **Different random init on each rank.** Fix: Seed identically before building the model (DDP also broadcasts rank 0's weights at wrap time, but wrapped-after-different-init is still a smell); use a different seed only for data augmentation.
- **Forgetting `DistributedSampler.set_epoch(epoch)`.** Fix: Call it at the start of every epoch or each epoch sees the same shuffled order.
- **Every rank saves/logs the checkpoint or prints.** Fix: Guard with `if dist.get_rank() == 0`; save `ddp.module.state_dict()` so keys have no `module.` prefix.
- **Conditional branches in `forward` leave some parameters without gradients.** Fix: DDP hangs or errors; fix the model or pass `find_unused_parameters=True` (slower).
- **Using DataParallel for multi-GPU training.** Fix: Switch to DDP: one process per GPU, via `torchrun`.
- **Treating the per-rank loss as the global loss.** Fix: `dist.all_reduce` the loss (then divide by world size) before logging a global number.
- **Effective batch size changes silently.** Fix: Global batch = per-rank batch x world_size; scale the learning rate (and warm up) accordingly.
- **Collectives called under different control flow on different ranks.** Fix: All ranks must call the same collectives in the same order, or the job deadlocks; set a `timeout` to surface hangs.
- **Spawning workers from a notebook cell with a locally-defined function.** Fix: Put the worker in a real `.py` file (as done here) and launch it via subprocess/torchrun.

## Exercises

**Exercise 1.** Modify the all-reduce script to compute the MEAN instead of the SUM (hint: divide by `world_size` after `all_reduce`) and run it with `run_script`. (Re-run the helper cells above first if needed.)

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
mean_src = '''
import os, torch, torch.distributed as dist, torch.multiprocessing as mp
def worker(rank, ws):
    os.environ["MASTER_ADDR"]="127.0.0.1"; os.environ["MASTER_PORT"]="29513"
    dist.init_process_group("gloo", rank=rank, world_size=ws)
    t = torch.tensor([float(rank + 1)])
    dist.all_reduce(t); t /= ws
    print(rank, t.item(), flush=True)
    dist.destroy_process_group()
if __name__ == "__main__":
    mp.spawn(worker, args=(2,), nprocs=2)
'''
import tempfile, subprocess, sys, os
p = os.path.join(tempfile.mkdtemp(), "m.py"); open(p, "w").write(mean_src)
print(subprocess.run([sys.executable, p], capture_output=True, text=True).stdout)   # both ranks print 1.5

0 1.5
1 1.5



**Exercise 2.** Use `DistributedSampler` with `num_replicas=4` on a dataset of 10 items. How many indices does each rank get, and why? (Print `len(sampler)`.)

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

10 is padded up to 12 by repeating samples so each rank gets ceil(10/4)=3 indices (use `drop_last=True` to trim instead).

In [14]:
# Solution 2
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler
ds = TensorDataset(torch.arange(10))
for r in range(4):
    s = DistributedSampler(ds, num_replicas=4, rank=r, shuffle=False)
    print(r, len(s), list(s))

0 3 [0, 4, 8]
1 3 [1, 5, 9]
2 3 [2, 6, 0]
3 3 [3, 7, 1]


**Exercise 3.** Compute FSDP per-rank state memory (16 bytes/param) for a 7B model on 8 ranks vs DDP.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

112 GB cannot fit on an 80 GB GPU with DDP, but 14 GB of sharded state can with FSDP.

In [16]:
# Solution 3
P = 7e9
print("DDP  GB/rank:", P*16/1e9)
print("FSDP GB/rank:", P*16/8/1e9)

DDP  GB/rank: 112.0
FSDP GB/rank: 14.0


**Exercise 4.** Show that averaging gradients of two equal-size shards equals the gradient of the full batch for a mean-reduced loss in a single process.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

This only holds for equal shard sizes and mean-reduced losses; with uneven shards you must weight by shard size.

In [18]:
# Solution 4
import torch.nn as nn
torch.manual_seed(0)
m = nn.Linear(3, 1); X = torch.randn(8, 3); Y = torch.randn(8, 1)
def grad(xs, ys):
    m.zero_grad(); nn.functional.mse_loss(m(xs), ys).backward()
    return m.weight.grad.clone()
full = grad(X, Y)
avg = (grad(X[:4], Y[:4]) + grad(X[4:], Y[4:])) / 2
print(torch.allclose(full, avg, atol=1e-6))

True


**Exercise 5.** Write a helper `is_main()` that returns True when not distributed or when rank is 0, and use it to guard a print.

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
import torch.distributed as dist
def is_main():
    return (not dist.is_available()) or (not dist.is_initialized()) or dist.get_rank() == 0
if is_main(): print("only rank 0 (or single process) prints this")

only rank 0 (or single process) prints this
